
# Attack 6 — Stealing-Guided Synonym Substitution

We propose a synonym substitution attack guided by statistical
differences between watermarked (WM) and unwatermarked (NoWM) texts.

The attack learns which words are more common in WM or NoWM
responses and uses WordNet to select replacement candidates.

We first implement a word-level approach, without using
LLaDA token statistics.

Our goal is to test whether learned word frequencies can
guide substitutions more effectively than random WordNet
substitution, without access to the watermark key.

## 1. Setup and Data Loading


In [1]:

import random
import re
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import nltk
from nltk.corpus import wordnet

def find_project_root(start: Path) -> Path:
    """Find the project root containing the official ETH repository."""
    for folder in [start, *start.parents]:
        if (folder / "diffusion-lm-watermark").is_dir():
            return folder
    raise FileNotFoundError("Project root not found.")

ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))

from detection import load_jsonl

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

nltk.download("wordnet", quiet=True)

wm = load_jsonl(
    ROOT / "results" / "Cleaned Data" / "watermarked_600.jsonl"
)
no_wm = load_jsonl(
    ROOT / "results" / "Cleaned Data" / "unwatermarked_600.jsonl"
)
baseline = load_jsonl(
    ROOT / "results" / "attacks" / "baseline_original.jsonl"
)

assert len(wm) == len(no_wm) == len(baseline) == 600
assert all(
    a["prompt"] == b["prompt"] == c["prompt"]
    for a, b, c in zip(wm, no_wm, baseline)
)

OUT_DIR = ROOT / "results" / "attacks" / "stealing_guided"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("WM responses:", len(wm))
print("NoWM responses:", len(no_wm))
print("Baseline responses:", len(baseline))
print("WordNet available:", len(wordnet.synsets("important")) > 0)


Project root: /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking
WM responses: 600
NoWM responses: 600
Baseline responses: 600
WordNet available: True



## 2. Train/Test Split

We split the 600 paired responses into 400 training pairs
and 200 test pairs.

The split is stratified by dataset and uses a fixed random
seed for reproducibility.

Training responses are used to learn word statistics.
Test responses are kept separate for the final attack evaluation.


In [2]:

from collections import defaultdict

groups = defaultdict(list)

for idx, row in enumerate(wm):
    dataset = row["dataset_name"].split("_batch_")[0]
    groups[dataset].append(idx)

rng = random.Random(SEED)

train_idx = []
test_idx = []

for dataset, indices in sorted(groups.items()):
    indices = indices.copy()
    rng.shuffle(indices)

    # 2/3 of each dataset for training
    n_train = 134 if dataset == sorted(groups)[0] else 133

    train_idx.extend(indices[:n_train])
    test_idx.extend(indices[n_train:])

    print(
        f"{dataset}: "
        f"{n_train} train / {len(indices) - n_train} test"
    )

train_idx.sort()
test_idx.sort()

assert len(train_idx) == 400
assert len(test_idx) == 200
assert set(train_idx).isdisjoint(test_idx)

print("\nTotal training pairs:", len(train_idx))
print("Total test pairs:", len(test_idx))
print("Originally detected in test:",
      sum(baseline[i]["detected"] for i in test_idx))


2-1_longform_qa: 134 train / 66 test
2-2_finance_qa: 133 train / 67 test
5-1_alpacafarm: 133 train / 67 test

Total training pairs: 400
Total test pairs: 200
Originally detected in test: 195



## 3. Learning Word Statistics

We estimate word frequencies using only the 400 training
pairs.

For each word, we compare its relative frequency in
watermarked and unwatermarked responses.

A smoothed log-ratio measures the association with
watermarked text. Positive scores indicate a stronger WM
association, while negative scores indicate a stronger
NoWM association.

These scores do not reveal the actual watermark greenlist.


In [3]:

print("WM fields:", list(wm[0].keys()))
print("NoWM fields:", list(no_wm[0].keys()))
print("Baseline fields:", list(baseline[0].keys()))


WM fields: ['prompt', 'completion', 'dataset_name', 'ages', 'length', 'z_score', 'p_value', 'token_color', 'mask', 'delta', 'convolution_kernel', 'topk', 'watermark_type', 'seeding_scheme', 'greenlist_type', 'greenlist_params', 'enforce_kl', 'n_iter', 'model_name', 'steps', 'gen_length', 'temperature', 'remasking', 'cfg_scale', 'mask_id', 'block_length']
NoWM fields: ['prompt', 'completion', 'dataset_name', 'ages', 'length', 'watermark_type', 'model_name', 'steps', 'gen_length', 'temperature', 'remasking', 'cfg_scale', 'mask_id', 'block_length']
Baseline fields: ['idx', 'dataset', 'prompt', 'p_value', 'detected', 'green_fraction', 'n_scored', 'n_tokens']


In [4]:

import math

# Simple word-level tokenization (no LLaDA tokenizer)
WORD_PATTERN = re.compile(r"[a-z]+(?:'[a-z]+)?")

def tokenize_words(text):
    return WORD_PATTERN.findall(text.lower())

# Count words in training data only
wm_counts = Counter()
no_wm_counts = Counter()

for i in train_idx:
    wm_counts.update(tokenize_words(wm[i]["completion"]))
    no_wm_counts.update(tokenize_words(no_wm[i]["completion"]))

total_wm = sum(wm_counts.values())
total_no_wm = sum(no_wm_counts.values())

# Smoothed log-ratio of relative word frequencies
ALPHA = 0.5
vocabulary = set(wm_counts) | set(no_wm_counts)
vocab_size = len(vocabulary)

word_scores = {}

for word in vocabulary:
    p_wm = (wm_counts[word] + ALPHA) / (
        total_wm + ALPHA * vocab_size
    )
    p_no_wm = (no_wm_counts[word] + ALPHA) / (
        total_no_wm + ALPHA * vocab_size
    )

    word_scores[word] = math.log(p_wm / p_no_wm)

print("Training WM words:", total_wm)
print("Training NoWM words:", total_no_wm)
print("Vocabulary size:", vocab_size)
print("Words scored:", len(word_scores))


Training WM words: 75365
Training NoWM words: 69168
Vocabulary size: 8715
Words scored: 8715


In [5]:

MIN_COUNT = 10

frequent_words = [
    word for word in vocabulary
    if wm_counts[word] + no_wm_counts[word] >= MIN_COUNT
]

print("=== TOP 20 WM-ASSOCIATED WORDS ===")

for word in sorted(
    frequent_words,
    key=lambda w: word_scores[w],
    reverse=True
)[:20]:
    print(
        f"{word:20s} "
        f"score={word_scores[word]:+6.3f} "
        f"WM={wm_counts[word]:4d} "
        f"NoWM={no_wm_counts[word]:4d}"
    )

print("\n=== TOP 20 NoWM-ASSOCIATED WORDS ===")

for word in sorted(
    frequent_words,
    key=lambda w: word_scores[w]
)[:20]:
    print(
        f"{word:20s} "
        f"score={word_scores[word]:+6.3f} "
        f"WM={wm_counts[word]:4d} "
        f"NoWM={no_wm_counts[word]:4d}"
    )


=== TOP 20 WM-ASSOCIATED WORDS ===
burst                score=+3.769 WM=  23 NoWM=   0
leader               score=+3.633 WM=  20 NoWM=   0
queen                score=+3.474 WM=  17 NoWM=   0
loop                 score=+3.353 WM=  15 NoWM=   0
sma                  score=+3.215 WM=  13 NoWM=   0
trick                score=+3.138 WM=  12 NoWM=   0
garnish              score=+3.055 WM=  11 NoWM=   0
elizabeth            score=+2.964 WM=  10 NoWM=   0
drawer               score=+2.534 WM=  20 NoWM=   1
die                  score=+2.254 WM=  15 NoWM=   1
mean                 score=+2.254 WM=  15 NoWM=   1
done                 score=+2.188 WM=  14 NoWM=   1
plans                score=+2.116 WM=  13 NoWM=   1
pop                  score=+2.039 WM=  12 NoWM=   1
pounds               score=+1.973 WM=  19 NoWM=   2
january              score=+1.956 WM=  11 NoWM=   1
society              score=+1.896 WM=  32 NoWM=   4
existing             score=+1.865 WM=  10 NoWM=   1
esteem               score=+1


## 4. Exploring Synonym Candidates

We use WordNet to find possible replacements for words
associated with watermarked text.

For each candidate, we compare its learned word-level
association score.

At this stage, we only inspect candidates. We do not
modify any test responses or use the watermark detector.

WordNet synonyms are not necessarily interchangeable
in context, which is a limitation of this first approach.


In [6]:

def get_synonyms(word):
    """Return single-word WordNet synonyms."""
    candidates = set()

    for synset in wordnet.synsets(word):
        for lemma in synset.lemmas():
            candidate = lemma.name().lower()

            if (
                candidate != word
                and re.fullmatch(r"[a-z]+", candidate)
            ):
                candidates.add(candidate)

    return sorted(candidates)


MIN_SUPPORT = 10

# Only consider words with enough training occurrences
eligible_words = [
    word for word in vocabulary
    if (
        wm_counts[word] + no_wm_counts[word] >= MIN_SUPPORT
        and word_scores[word] > 0
    )
]

examples = []

for word in eligible_words:
    synonyms = get_synonyms(word)

    # Keep synonyms with enough statistical evidence
    candidates = [
        synonym for synonym in synonyms
        if (
            wm_counts[synonym] + no_wm_counts[synonym]
            >= MIN_SUPPORT
            and word_scores.get(synonym, 0) < word_scores[word]
        )
    ]

    if candidates:
        best = min(candidates, key=lambda s: word_scores[s])

        examples.append({
            "original": word,
            "original_score": word_scores[word],
            "replacement": best,
            "replacement_score": word_scores[best],
            "gain": word_scores[word] - word_scores[best],
        })

examples.sort(key=lambda x: x["gain"], reverse=True)

print("Eligible WM-associated words:", len(eligible_words))
print("Words with supported better synonyms:", len(examples))

print("\n=== TOP 20 POTENTIAL SUBSTITUTIONS ===")

for ex in examples[:20]:
    print(
        f"{ex['original']:18s} -> "
        f"{ex['replacement']:18s} "
        f"WM score: {ex['original_score']:+.2f} -> "
        f"{ex['replacement_score']:+.2f} "
        f"(gain: {ex['gain']:.2f})"
    )


Eligible WM-associated words: 989
Words with supported better synonyms: 554

=== TOP 20 POTENTIAL SUBSTITUTIONS ===
burst              -> fit                WM score: +3.77 -> -1.08 (gain: 4.85)
leaves             -> leaf               WM score: +0.16 -> -3.45 (gain: 3.60)
queen              -> king               WM score: +3.47 -> -0.08 (gain: 3.56)
trick              -> magic              WM score: +3.14 -> -0.38 (gain: 3.52)
tried              -> hear               WM score: +1.35 -> -1.93 (gain: 3.28)
compressing        -> press              WM score: +1.14 -> -2.03 (gain: 3.17)
mean               -> average            WM score: +2.25 -> -0.74 (gain: 3.00)
fighting           -> push               WM score: +0.81 -> -2.12 (gain: 2.92)
described          -> line               WM score: +0.68 -> -2.19 (gain: 2.87)
course             -> line               WM score: +0.67 -> -2.19 (gain: 2.85)
die                -> exit               WM score: +2.25 -> -0.59 (gain: 2.85)
learned        


## 5. Stealing-Guided Synonym Substitution

We build a substitution dictionary from the learned
WM/NoWM word statistics.

For a given text, we prioritize words whose replacement
produces the largest decrease in WM association score.

The attack initially targets 10% of the words.

We first run a pilot on 10 training responses to verify
the implementation before evaluating held-out examples.

Semantic preservation is not enforced in this first version.


In [8]:

# Dictionary built exclusively from training statistics
substitution_dict = {
    ex["original"]: {
        "replacement": ex["replacement"],
        "gain": ex["gain"],
    }
    for ex in examples
}

print("Dictionary size:", len(substitution_dict))

# Example
for word in ["important", "mean", "energy", "society"]:
    print(word, "->", substitution_dict.get(word))


Dictionary size: 554
important -> None
mean -> {'replacement': 'average', 'gain': 2.9986691332273008}
energy -> {'replacement': 'push', 'gain': 2.836409511379557}
society -> {'replacement': 'company', 'gain': 2.115313031040235}


In [9]:

def stealing_guided_attack(text, ratio=0.10):
    matches = list(WORD_PATTERN.finditer(text.lower()))

    # Select candidates using the learned dictionary
    candidates = []

    for i, match in enumerate(matches):
        word = match.group()

        if word in substitution_dict:
            candidates.append({
                "index": i,
                "start": match.start(),
                "end": match.end(),
                "original": word,
                "replacement": substitution_dict[word]["replacement"],
                "gain": substitution_dict[word]["gain"],
            })

    # Highest expected statistical gain first
    candidates.sort(key=lambda x: x["gain"], reverse=True)

    target = round(len(matches) * ratio)
    selected = candidates[:target]

    # Replace from right to left to preserve character positions
    attacked = text

    for item in sorted(
        selected,
        key=lambda x: x["start"],
        reverse=True
    ):
        original_surface = attacked[item["start"]:item["end"]]
        replacement = item["replacement"]

        # Preserve basic capitalization
        if original_surface.isupper():
            replacement = replacement.upper()
        elif original_surface.istitle():
            replacement = replacement.capitalize()

        attacked = (
            attacked[:item["start"]]
            + replacement
            + attacked[item["end"]:]
        )

    return {
        "attacked": attacked,
        "n_words": len(matches),
        "n_targeted": target,
        "n_replaced": len(selected),
        "substitutions": selected,
    }



## 6. Pilot Test

We apply the proposed attack to 10 watermarked training
responses with a 10% word replacement budget.

This pilot verifies substitution coverage and basic
implementation behavior.

No held-out test responses or watermark detection scores
are used at this stage.



In [10]:

pilot_indices = random.Random(SEED + 1).sample(train_idx, 10)

pilot_results = []

for idx in pilot_indices:
    original = wm[idx]["completion"]
    result = stealing_guided_attack(original, ratio=0.10)

    pilot_results.append(result)

    print(f"\n{'=' * 60}")
    print(f"Response index: {idx}")
    print(f"Words: {result['n_words']}")
    print(f"Targeted: {result['n_targeted']}")
    print(f"Replaced: {result['n_replaced']}")

    print("\nSubstitutions:")
    for sub in result["substitutions"][:10]:
        print(
            f"  {sub['original']:15s} -> "
            f"{sub['replacement']:15s} "
            f"(gain={sub['gain']:.2f})"
        )

print(f"\n{'=' * 60}")
print("PILOT SUMMARY")

total_words = sum(r["n_words"] for r in pilot_results)
total_targeted = sum(r["n_targeted"] for r in pilot_results)
total_replaced = sum(r["n_replaced"] for r in pilot_results)

print("Total words:", total_words)
print("Targeted replacements:", total_targeted)
print("Actual replacements:", total_replaced)
print(f"Actual replacement ratio: {total_replaced / total_words:.1%}")



Response index: 33
Words: 247
Targeted: 25
Replaced: 25

Substitutions:
  learned         -> hear            (gain=2.84)
  learned         -> hear            (gain=2.84)
  learned         -> hear            (gain=2.84)
  made            -> fix             (gain=1.32)
  made            -> fix             (gain=1.32)
  made            -> fix             (gain=1.32)
  made            -> fix             (gain=1.32)
  let             -> allow           (gain=1.30)
  came            -> follow          (gain=1.22)
  out             -> extinct         (gain=1.13)

Response index: 220
Words: 41
Targeted: 4
Replaced: 4

Substitutions:
  reported        -> cover           (gain=1.65)
  purposes        -> aim             (gain=1.27)
  release         -> loss            (gain=1.15)
  can             -> bottom          (gain=1.12)

Response index: 538
Words: 256
Targeted: 26
Replaced: 26

Substitutions:
  learning        -> hear            (gain=2.49)
  society         -> company         (gain=2.12


## 7. Random WordNet Baseline

We compare our statistically guided synonym selection against
random WordNet synonym selection at identical word positions.

This controlled baseline isolates the benefit of selecting
a synonym using learned WM/NoWM statistics.

Both approaches modify the same number of words.

Semantic preservation is not enforced.


In [13]:

def random_wordnet_attack(text, guided_result, seed=42):
    """Replace the same positions with random WordNet synonyms."""
    rng = random.Random(seed)
    attacked = text

    substitutions = []

    for item in sorted(
        guided_result["substitutions"],
        key=lambda x: x["start"],
        reverse=True
    ):
        original = item["original"]
        synonyms = get_synonyms(original)

        if not synonyms:
            raise ValueError(f"No WordNet synonyms for {original}")

        replacement = rng.choice(synonyms)

        surface = attacked[item["start"]:item["end"]]

        if surface.isupper():
            replacement = replacement.upper()
        elif surface.istitle():
            replacement = replacement.capitalize()

        attacked = (
            attacked[:item["start"]]
            + replacement
            + attacked[item["end"]:]
        )

        substitutions.append({
            "original": original,
            "replacement": replacement,
        })

    return {
        "attacked": attacked,
        "n_words": guided_result["n_words"],
        "n_replaced": len(substitutions),
        "substitutions": substitutions,
    }


# Pilot comparison on the same 10 training responses


for idx in pilot_indices[:3]:
    original = wm[idx]["completion"]

    guided = stealing_guided_attack(original, ratio=0.10)
    random_result = random_wordnet_attack(
        original, guided, seed=SEED + idx
    )

    guided_sorted = sorted(
        guided["substitutions"],
        key=lambda x: x["start"]
    )

    random_sorted = list(reversed(
        random_result["substitutions"]
    ))

    print(f"\nResponse {idx}")
    print("Replacements:", guided["n_replaced"])

    for g, r in zip(guided_sorted[:5], random_sorted[:5]):
        assert g["original"] == r["original"]

        print(
            f"{g['original']:15s} "
            f"Guided: {g['replacement']:15s} "
            f"Random: {r['replacement']}"
        )





Response 33
Replacements: 25
let             Guided: allow           Random: lease
tell            Guided: evidence        Random: secernate
came            Guided: follow          Random: number
lived           Guided: exist           Random: know
big             Guided: heavy           Random: bragging

Response 220
Replacements: 4
reported        Guided: cover           Random: report
release         Guided: loss            Random: sack
purposes        Guided: aim             Random: purpose
can             Guided: bottom          Random: john

Response 538
Replacements: 26
speaking        Guided: talk            Random: talk
tell            Guided: evidence        Random: secernate
thoughts        Guided: thinking        Random: sentiment
make            Guided: fix             Random: produce
do              Guided: manage          Random: brawl



## 8. Held-Out Attack Generation

We apply the stealing-guided and random WordNet attacks
to the 200 held-out watermarked responses.

Both methods use the same substitution positions and
replacement budget (10%).

The test texts were never used to learn word statistics
or construct the substitution dictionary.


In [14]:

import json

RATIO = 0.10
guided_records = []
random_records = []

for idx in test_idx:
    original = wm[idx]["completion"]

    guided = stealing_guided_attack(original, ratio=RATIO)
    random_result = random_wordnet_attack(
        original, guided, seed=SEED + idx
    )

    assert guided["n_replaced"] == random_result["n_replaced"]

    common = {
        "idx": idx,
        "dataset": wm[idx]["dataset_name"],
        "prompt": wm[idx]["prompt"],
        "original": original,
        "n_words": guided["n_words"],
        "n_targeted": guided["n_targeted"],
        "n_replaced": guided["n_replaced"],
        "ratio": RATIO,
    }

    guided_records.append({
        **common,
        "attack": "stealing_guided",
        "attacked": guided["attacked"],
        "substitutions": guided["substitutions"],
    })

    random_records.append({
        **common,
        "attack": "random_wordnet_same_positions",
        "attacked": random_result["attacked"],
        "substitutions": random_result["substitutions"],
    })


def save_attack_records(path, records):
    if path.exists():
        raise FileExistsError(f"File already exists: {path}")

    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


guided_path = OUT_DIR / "stealing_guided_r0.1.jsonl"
random_path = OUT_DIR / "random_wordnet_r0.1.jsonl"

save_attack_records(guided_path, guided_records)
save_attack_records(random_path, random_records)

print("Guided texts:", len(guided_records))
print("Random texts:", len(random_records))

print("Total replacements:", sum(
    r["n_replaced"] for r in guided_records
))

print("Mean replacement ratio:", round(
    sum(r["n_replaced"] for r in guided_records)
    / sum(r["n_words"] for r in guided_records),
    4
))

print("Saved to:", OUT_DIR)


Guided texts: 200
Random texts: 200
Total replacements: 3769
Mean replacement ratio: 0.0985
Saved to: /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided



## 9. Watermark Detection

We evaluate the 200 held-out responses using the same
local watermark detector as in previous experiments.

The attack success rate (ASR) is computed only over
responses that were originally detected as watermarked.

The detector is used for evaluation only, not for
selecting substitutions.


In [15]:

import inspect
from detection import load_detector, detect

print("load_detector:", inspect.signature(load_detector))
print("detect:", inspect.signature(detect))

print("\nSample baseline:")
print(baseline[test_idx[0]])

print("\nSaved attacks:")
print("Guided:", guided_path.exists(), len(guided_records))
print("Random:", random_path.exists(), len(random_records))

assert len(guided_records) == len(random_records) == 200
assert all(
    g["idx"] == r["idx"]
    and g["n_replaced"] == r["n_replaced"]
    for g, r in zip(guided_records, random_records)
)

print("\nAll checks passed!")


load_detector: (root: pathlib.Path)
detect: (text: str, tokenizer, detector) -> dict

Sample baseline:
{'idx': 1, 'dataset': '2-1_longform_qa', 'prompt': "user\n\nYou are a helpful assistant, please answer the following question within 300 words:\n\nhow do we diferentiate the frequency of a note from the sound that characterize a certain instirment ? Why can I tell an A on a cello apart from an A on a piano ? Please explain like I'm five.assistant\n\n", 'p_value': 2.4346417515420958e-09, 'detected': True, 'green_fraction': 0.42857142857142855, 'n_scored': 231, 'n_tokens': 298}

Saved attacks:
Guided: True 200
Random: True 200

All checks passed!


In [16]:

from tqdm.auto import tqdm
from detection import load_detector, detect, save_jsonl

# Load the same detector as in previous experiments
tokenizer, detector = load_detector(ROOT)

def evaluate_attack(records, name):
    results = []

    for record in tqdm(records, desc=name):
        detection = detect(
            record["attacked"],
            tokenizer,
            detector
        )

        results.append({
            "idx": record["idx"],
            "dataset": record["dataset"],
            "attack": name,
            "ratio": record["ratio"],
            "n_words": record["n_words"],
            "n_replaced": record["n_replaced"],
            "original_detected": baseline[record["idx"]]["detected"],
            "original_p_value": baseline[record["idx"]]["p_value"],
            **detection,
        })

    return results


guided_detection = evaluate_attack(
    guided_records, "stealing_guided"
)

random_detection = evaluate_attack(
    random_records, "random_wordnet_same_positions"
)

print("Guided evaluations:", len(guided_detection))
print("Random evaluations:", len(random_detection))


/Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
random_wordnet_same_positions: 100%|██████████| 200/200 [00:00<00:00, 1346.01it/s]

Guided evaluations: 200
Random evaluations: 200


In [18]:

guided_detection_path = OUT_DIR / "stealing_guided_detection_r0.1.jsonl"
random_detection_path = OUT_DIR / "random_wordnet_detection_r0.1.jsonl"

# Refuse accidental overwrite
assert not guided_detection_path.exists()
assert not random_detection_path.exists()

save_jsonl(guided_detection, guided_detection_path)
save_jsonl(random_detection, random_detection_path)

print("Saved:", guided_detection_path.name)
print("Saved:", random_detection_path.name)


Saved 200 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided/stealing_guided_detection_r0.1.jsonl
Saved 200 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided/random_wordnet_detection_r0.1.jsonl
Saved: stealing_guided_detection_r0.1.jsonl
Saved: random_wordnet_detection_r0.1.jsonl


In [19]:

def summarize_attack(results):
    originally_detected = [
        r for r in results if r["original_detected"]
    ]

    n = len(originally_detected)

    successes = sum(
        not r["detected"]
        for r in originally_detected
    )

    return {
        "evaluated": len(results),
        "originally_detected": n,
        "successful_attacks": successes,
        "asr": successes / n if n else 0,
        "detected_after": sum(r["detected"] for r in results),
    }


guided_summary = summarize_attack(guided_detection)
random_summary = summarize_attack(random_detection)

print("=== HELD-OUT TEST RESULTS (10% substitutions) ===")

for name, summary in [
    ("Stealing-Guided", guided_summary),
    ("Random WordNet", random_summary),
]:
    print(f"\n{name}")
    print("Originally detected:", summary["originally_detected"])
    print("Successful attacks:", summary["successful_attacks"])
    print(f"ASR: {summary['asr']:.1%}")
    print("Detected after attack:", summary["detected_after"])

difference = (
    guided_summary["asr"] - random_summary["asr"]
)

print(f"\nASR difference (Guided - Random): {difference:+.1%}")


=== HELD-OUT TEST RESULTS (10% substitutions) ===

Stealing-Guided
Originally detected: 195
Successful attacks: 13
ASR: 6.7%
Detected after attack: 182

Random WordNet
Originally detected: 195
Successful attacks: 17
ASR: 8.7%
Detected after attack: 179

ASR difference (Guided - Random): -2.1%



## 12. Comparison with Reproduced WordNet Attack

We compare three synonym substitution strategies:

1. Reproduced WordNet substitution (MarkLLM)
2. Stealing-Guided substitution (our method)
3. Random WordNet substitution at guided-selected positions

All methods are evaluated on the same 200 held-out responses.

Attack success rate (ASR) is calculated over the 195
responses originally detected as watermarked.

We also compare the actual number of word replacements,
since equal nominal attack ratios may not imply equal
perturbation intensity.


In [20]:

syn_dir = ROOT / "results" / "attacks" / "synonym_substitution"

print("Available files:")
for path in sorted(syn_dir.glob("*.jsonl")):
    print(" -", path.name)

syn_path = syn_dir / "synonym_substitution_r0.1.jsonl"

assert syn_path.exists(), f"Missing file: {syn_path}"

syn_records = load_jsonl(syn_path)

print("\nRecords:", len(syn_records))
print("Fields:", list(syn_records[0].keys()))

print("\nFirst record:")
for key, value in syn_records[0].items():
    if isinstance(value, str) and len(value) > 120:
        print(f"{key}: {value[:120]}...")
    else:
        print(f"{key}: {value}")


Available files:
 - synonym_substitution_r0.1.jsonl
 - synonym_substitution_r0.2.jsonl
 - synonym_substitution_r0.3.jsonl
 - synonym_substitution_r0.4.jsonl
 - synonym_substitution_r0.5.jsonl

Records: 600
Fields: ['idx', 'dataset', 'attack', 'ratio', 'seed', 'markllm_version', 'prompt', 'original', 'attacked', 'n_words_original', 'n_words_attacked', 'n_replaceable', 'n_replaced', 'original_p_value', 'original_detected', 'original_green_fraction', 'p_value', 'detected', 'green_fraction', 'n_scored', 'n_tokens']

First record:
idx: 0
dataset: 2-1_longform_qa
attack: synonym_substitution
ratio: 0.1
seed: 42
markllm_version: 0.1.5
prompt: user

You are a helpful assistant, please answer the following question within 300 words:

( LI5 ) Explain to me the col...
original: The collapse of the Soviet Union is when one of the largest superpowers, like a big super-strong robot that had been fig...
attacked: The collapse of the Soviet Union is when unitary of the largest superpowers, like type A

In [21]:

# Build lookup tables by original response index
syn_by_idx = {r["idx"]: r for r in syn_records}
guided_by_idx = {r["idx"]: r for r in guided_detection}
random_by_idx = {r["idx"]: r for r in random_detection}

assert len(syn_by_idx) == 600
assert len(guided_by_idx) == len(random_by_idx) == 200
assert set(test_idx) == set(guided_by_idx) == set(random_by_idx)

# Check that the reproduced attack uses the same original texts
for idx in test_idx:
    assert syn_by_idx[idx]["original"] == wm[idx]["completion"]
    assert syn_by_idx[idx]["original_detected"] == baseline[idx]["detected"]

# Evaluate all methods on the SAME originally detected responses
eligible_idx = [
    idx for idx in test_idx
    if baseline[idx]["detected"]
]

assert len(eligible_idx) == 195

methods = {
    "Reproduced WordNet": syn_by_idx,
    "Stealing-Guided V1": guided_by_idx,
    "Random (same positions)": random_by_idx,
}

print("=== FAIR COMPARISON — 10% TARGET RATIO ===")
print(f"Test texts: {len(test_idx)}")
print(f"Originally detected: {len(eligible_idx)}\n")

for name, records in methods.items():
    successes = sum(
        not records[idx]["detected"]
        for idx in eligible_idx
    )

    asr = successes / len(eligible_idx)

    mean_green = np.mean([
        records[idx]["green_fraction"]
        for idx in eligible_idx
    ])

    print(name)
    print(f"  Successful attacks: {successes}/{len(eligible_idx)}")
    print(f"  ASR: {asr:.2%}")
    print(f"  Mean green fraction after attack: {mean_green:.4f}")
    print()


=== FAIR COMPARISON — 10% TARGET RATIO ===
Test texts: 200
Originally detected: 195

Reproduced WordNet
  Successful attacks: 6/195
  ASR: 3.08%
  Mean green fraction after attack: 0.4162

Stealing-Guided V1
  Successful attacks: 13/195
  ASR: 6.67%
  Mean green fraction after attack: 0.4023

Random (same positions)
  Successful attacks: 17/195
  ASR: 8.72%
  Mean green fraction after attack: 0.3963



In [22]:

print("=== SUBSTITUTION BUDGET COMPARISON ===")

for name, records in methods.items():
    total_targeted = sum(
        records[idx]["n_replaced"]
        for idx in test_idx
    )

    print(f"{name}: {total_targeted} reported replacements")

print("\nIMPORTANT:")
print("For reproduced WordNet, n_replaced may be an estimate.")
print("For Guided and Random, replacements are explicitly counted.")


=== SUBSTITUTION BUDGET COMPARISON ===
Reproduced WordNet: 3811 reported replacements
Stealing-Guided V1: 3769 reported replacements
Random (same positions): 3769 reported replacements

IMPORTANT:
For reproduced WordNet, n_replaced may be an estimate.
For Guided and Random, replacements are explicitly counted.



## 13. Fully Random WordNet Baseline

We implement a fourth attack that randomly selects both
word positions and WordNet synonyms.

For each held-out response, the number of substitutions
is matched to our stealing-guided attack.

This baseline helps isolate the effect of guided word
position selection from the effect of guided synonym selection.

Semantic preservation is not enforced.


In [23]:

def fully_random_wordnet_attack(text, n_replacements, seed=42):
    rng = random.Random(seed)
    matches = list(WORD_PATTERN.finditer(text.lower()))

    candidates = []

    for match in matches:
        word = match.group()
        synonyms = get_synonyms(word)

        if synonyms:
            candidates.append({
                "start": match.start(),
                "end": match.end(),
                "original": word,
                "synonyms": synonyms,
            })

    if len(candidates) < n_replacements:
        raise ValueError(
            f"Only {len(candidates)} replaceable positions, "
            f"but {n_replacements} requested."
        )

    selected = rng.sample(candidates, n_replacements)
    attacked = text
    substitutions = []

    for item in sorted(
        selected, key=lambda x: x["start"], reverse=True
    ):
        replacement = rng.choice(item["synonyms"])
        surface = attacked[item["start"]:item["end"]]

        if surface.isupper():
            replacement = replacement.upper()
        elif surface.istitle():
            replacement = replacement.capitalize()

        attacked = (
            attacked[:item["start"]]
            + replacement
            + attacked[item["end"]:]
        )

        substitutions.append({
            "original": item["original"],
            "replacement": replacement,
            "start": item["start"],
            "end": item["end"],
        })

    return {
        "attacked": attacked,
        "n_words": len(matches),
        "n_replaced": len(substitutions),
        "substitutions": substitutions,
    }


In [24]:

fully_random_records = []

for guided_record in guided_records:
    idx = guided_record["idx"]

    result = fully_random_wordnet_attack(
        text=guided_record["original"],
        n_replacements=guided_record["n_replaced"],
        seed=SEED + idx + 1000,
    )

    assert result["n_replaced"] == guided_record["n_replaced"]

    fully_random_records.append({
        "idx": idx,
        "dataset": guided_record["dataset"],
        "prompt": guided_record["prompt"],
        "original": guided_record["original"],
        "attacked": result["attacked"],
        "attack": "fully_random_wordnet",
        "ratio": RATIO,
        "n_words": result["n_words"],
        "n_replaced": result["n_replaced"],
        "substitutions": result["substitutions"],
    })

fully_random_path = OUT_DIR / "fully_random_wordnet_r0.1.jsonl"

assert not fully_random_path.exists()
save_jsonl(fully_random_records, fully_random_path)

print("Texts:", len(fully_random_records))
print("Total replacements:", sum(
    r["n_replaced"] for r in fully_random_records
))
print("Saved:", fully_random_path.name)


Saved 200 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided/fully_random_wordnet_r0.1.jsonl
Texts: 200
Total replacements: 3769
Saved: fully_random_wordnet_r0.1.jsonl


In [25]:

fully_random_detection = evaluate_attack(
    fully_random_records,
    "fully_random_wordnet"
)

fully_random_detection_path = (
    OUT_DIR / "fully_random_wordnet_detection_r0.1.jsonl"
)

assert not fully_random_detection_path.exists()
save_jsonl(
    fully_random_detection,
    fully_random_detection_path
)

print("Saved:", fully_random_detection_path.name)


fully_random_wordnet: 100%|██████████| 200/200 [00:00<00:00, 937.14it/s]

Saved 200 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided/fully_random_wordnet_detection_r0.1.jsonl
Saved: fully_random_wordnet_detection_r0.1.jsonl


In [26]:

all_methods = {
    "Reproduced WordNet": syn_by_idx,
    "Stealing-Guided V1": guided_by_idx,
    "Random synonyms, guided positions": random_by_idx,
    "Fully Random WordNet": {
        r["idx"]: r for r in fully_random_detection
    },
}

print("=== FOUR-METHOD COMPARISON ===")
print("Originally detected:", len(eligible_idx))

for name, records in all_methods.items():
    successes = sum(
        not records[idx]["detected"]
        for idx in eligible_idx
    )

    print(
        f"{name:38s} "
        f"{successes:3d}/{len(eligible_idx)} "
        f"ASR={successes / len(eligible_idx):.2%}"
    )


=== FOUR-METHOD COMPARISON ===
Originally detected: 195
Reproduced WordNet                       6/195 ASR=3.08%
Stealing-Guided V1                      13/195 ASR=6.67%
Random synonyms, guided positions       17/195 ASR=8.72%
Fully Random WordNet                     9/195 ASR=4.62%


In [27]:

# Audit: do guided synonyms really favor NoWM?

total = 0
negative_score = 0
positive_score = 0
zero_score = 0

for record in guided_records:
    for sub in record["substitutions"]:
        synonym = sub["replacement"]
        score = word_scores[synonym]

        total += 1

        if score < 0:
            negative_score += 1
        elif score > 0:
            positive_score += 1
        else:
            zero_score += 1

print("=== GUIDED SYNONYM AUDIT ===")
print("Total substitutions:", total)
print(f"NoWM-associated synonyms: {negative_score} ({negative_score/total:.1%})")
print(f"WM-associated synonyms: {positive_score} ({positive_score/total:.1%})")
print(f"Neutral synonyms: {zero_score} ({zero_score/total:.1%})")


=== GUIDED SYNONYM AUDIT ===
Total substitutions: 3769
NoWM-associated synonyms: 3669 (97.3%)
WM-associated synonyms: 100 (2.7%)
Neutral synonyms: 0 (0.0%)



## 14. Multi-Seed Robustness Analysis

We evaluate two random WordNet baselines using 10
independent random seeds.

Both baselines use the same held-out test set and
the same number of substitutions per response.

We report mean ASR and standard deviation across seeds.

The stealing-guided attack is deterministic and is
included as a fixed reference.


In [28]:

SEEDS = list(range(100, 110))

multiseed_results = []

for seed in SEEDS:
    print(f"\n=== Seed {seed} ===")

    for method in [
        "random_synonyms_guided_positions",
        "fully_random_wordnet",
    ]:
        records = []

        for guided_record in guided_records:
            idx = guided_record["idx"]
            original = guided_record["original"]

            if method == "random_synonyms_guided_positions":
                guided = stealing_guided_attack(original, ratio=RATIO)

                result = random_wordnet_attack(
                    original,
                    guided,
                    seed=seed * 1000 + idx
                )
            else:
                result = fully_random_wordnet_attack(
                    original,
                    n_replacements=guided_record["n_replaced"],
                    seed=seed * 1000 + idx
                )

            records.append({
                "idx": idx,
                "dataset": guided_record["dataset"],
                "attack": method,
                "ratio": RATIO,
                "n_words": guided_record["n_words"],
                "n_replaced": result["n_replaced"],
                "attacked": result["attacked"],
            })

        detections = evaluate_attack(records, method)

        eligible = [
            r for r in detections if r["original_detected"]
        ]

        successes = sum(not r["detected"] for r in eligible)
        asr = successes / len(eligible)

        multiseed_results.append({
            "seed": seed,
            "method": method,
            "successes": successes,
            "eligible": len(eligible),
            "asr": asr,
        })

        print(f"{method}: ASR={asr:.2%}")



=== Seed 100 ===


random_synonyms_guided_positions: 100%|██████████| 200/200 [00:00<00:00, 1175.88it/s]

random_synonyms_guided_positions: ASR=6.15%



fully_random_wordnet: 100%|██████████| 200/200 [00:00<00:00, 1009.47it/s]


fully_random_wordnet: ASR=5.13%

=== Seed 101 ===


random_synonyms_guided_positions: 100%|██████████| 200/200 [00:00<00:00, 1364.55it/s]


... (31 outputs left out)

random_synonyms_guided_positions: 100%|██████████| 200/200 [00:00<00:00, 1347.75it/s]


random_synonyms_guided_positions: ASR=4.62%


fully_random_wordnet: 100%|██████████| 200/200 [00:00<00:00, 1325.72it/s]

fully_random_wordnet: ASR=4.10%


In [29]:

multiseed_path = OUT_DIR / "multiseed_results_r0.1.jsonl"

assert not multiseed_path.exists()
save_jsonl(multiseed_results, multiseed_path)

print("Saved:", multiseed_path)
print("Results:", len(multiseed_results))


Saved 20 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided/multiseed_results_r0.1.jsonl
Saved: /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided/multiseed_results_r0.1.jsonl
Results: 20


In [30]:

print("=== MULTI-SEED RESULTS ===")

for method in [
    "random_synonyms_guided_positions",
    "fully_random_wordnet",
]:
    values = np.array([
        r["asr"]
        for r in multiseed_results
        if r["method"] == method
    ])

    print(f"\n{method}")
    print(f"Mean ASR: {values.mean():.2%}")
    print(f"Std ASR:  {values.std(ddof=1):.2%}")
    print(f"Min ASR:  {values.min():.2%}")
    print(f"Max ASR:  {values.max():.2%}")

print("\nStealing-Guided V1 (fixed): 6.67%")


=== MULTI-SEED RESULTS ===

random_synonyms_guided_positions
Mean ASR: 6.82%
Std ASR:  1.30%
Min ASR:  4.62%
Max ASR:  8.72%

fully_random_wordnet
Mean ASR: 3.90%
Std ASR:  1.14%
Min ASR:  2.05%
Max ASR:  5.13%

Stealing-Guided V1 (fixed): 6.67%



## 15. Multi-Split Robustness

We evaluate the stability of the proposed attack across
10 different stratified train/test splits.

For each split, we independently rebuild word frequencies,
association scores, and the substitution dictionary using
400 paired training examples.

All attack methods are evaluated on the same 200 held-out
responses within each split.

No held-out responses are used to build the dictionaries
or select attack candidates.


In [31]:

from functools import lru_cache

SPLIT_SEEDS = [42, 100, 101, 102, 103, 104, 105, 106, 107, 108]

@lru_cache(maxsize=None)
def cached_synonyms(word):
    return tuple(get_synonyms(word))


def make_split(seed):
    rng = random.Random(seed)
    train, test = [], []

    for j, (dataset, original_indices) in enumerate(
        sorted(groups.items())
    ):
        indices = original_indices.copy()
        rng.shuffle(indices)

        n_train = 134 if j == 0 else 133

        train.extend(indices[:n_train])
        test.extend(indices[n_train:])

    return sorted(train), sorted(test)


def learn_dictionary(training_indices):
    wm_c = Counter()
    no_wm_c = Counter()

    for i in training_indices:
        wm_c.update(tokenize_words(wm[i]["completion"]))
        no_wm_c.update(tokenize_words(no_wm[i]["completion"]))

    vocab = set(wm_c) | set(no_wm_c)
    v = len(vocab)

    total_wm = sum(wm_c.values())
    total_no_wm = sum(no_wm_c.values())

    scores = {}

    for word in vocab:
        p_wm = (wm_c[word] + ALPHA) / (
            total_wm + ALPHA * v
        )
        p_no_wm = (no_wm_c[word] + ALPHA) / (
            total_no_wm + ALPHA * v
        )
        scores[word] = math.log(p_wm / p_no_wm)

    dictionary = {}

    for word in vocab:
        if wm_c[word] + no_wm_c[word] < MIN_SUPPORT:
            continue
        if scores[word] <= 0:
            continue

        candidates = [
            synonym for synonym in cached_synonyms(word)
            if (
                wm_c[synonym] + no_wm_c[synonym] >= MIN_SUPPORT
                and scores.get(synonym, float("inf")) < scores[word]
            )
        ]

        if candidates:
            best = min(candidates, key=lambda s: scores[s])

            dictionary[word] = {
                "replacement": best,
                "gain": scores[word] - scores[best],
            }

    return dictionary


def guided_attack_with_dictionary(text, dictionary, ratio=0.10):
    matches = list(WORD_PATTERN.finditer(text.lower()))
    candidates = []

    for match in matches:
        word = match.group()

        if word in dictionary:
            candidates.append({
                "start": match.start(),
                "end": match.end(),
                "original": word,
                "replacement": dictionary[word]["replacement"],
                "gain": dictionary[word]["gain"],
            })

    candidates.sort(key=lambda x: x["gain"], reverse=True)

    target = round(len(matches) * ratio)
    selected = candidates[:target]
    attacked = text

    for item in sorted(selected, key=lambda x: x["start"], reverse=True):
        surface = attacked[item["start"]:item["end"]]
        replacement = item["replacement"]

        if surface.isupper():
            replacement = replacement.upper()
        elif surface.istitle():
            replacement = replacement.capitalize()

        attacked = (
            attacked[:item["start"]]
            + replacement
            + attacked[item["end"]:]
        )

    return {
        "attacked": attacked,
        "n_words": len(matches),
        "n_replaced": len(selected),
        "substitutions": selected,
    }


# Validate seed 42 against our original experiment
train_42, test_42 = make_split(42)
dictionary_42 = learn_dictionary(train_42)

assert train_42 == train_idx
assert test_42 == test_idx

assert dictionary_42 == substitution_dict

print("Original V1 dictionary:", len(substitution_dict))
print("Rebuilt seed 42 dictionary:", len(dictionary_42))
print("Original split reproduced:", True)


Original V1 dictionary: 554
Rebuilt seed 42 dictionary: 554
Original split reproduced: True


In [32]:

split_results = []
split_details = []

for split_seed in SPLIT_SEEDS:
    train_ids, test_ids = make_split(split_seed)
    dictionary = learn_dictionary(train_ids)

    print(
        f"\nSplit {split_seed} | "
        f"Dictionary: {len(dictionary)} entries"
    )

    attacks = {
        "guided": [],
        "random_synonyms_guided_positions": [],
        "fully_random": [],
    }

    for idx in test_ids:
        original = wm[idx]["completion"]

        guided = guided_attack_with_dictionary(
            original, dictionary, ratio=0.10
        )

        random_syn = random_wordnet_attack(
            original,
            guided,
            seed=split_seed * 1000 + idx
        )

        fully_random = fully_random_wordnet_attack(
            original,
            n_replacements=guided["n_replaced"],
            seed=split_seed * 1000 + idx + 500000
        )

        for name, result in [
            ("guided", guided),
            ("random_synonyms_guided_positions", random_syn),
            ("fully_random", fully_random),
        ]:
            attacks[name].append({
                "idx": idx,
                "dataset": wm[idx]["dataset_name"],
                "attack": name,
                "ratio": 0.10,
                "n_words": result["n_words"],
                "n_replaced": result["n_replaced"],
                "attacked": result["attacked"],
            })

    for name, records in attacks.items():
        detections = evaluate_attack(
            records, f"{split_seed}: {name}"
        )

        eligible = [
            r for r in detections if r["original_detected"]
        ]

        successes = sum(not r["detected"] for r in eligible)

        summary = {
            "split_seed": split_seed,
            "method": name,
            "dictionary_size": len(dictionary),
            "test_count": len(test_ids),
            "eligible": len(eligible),
            "successes": successes,
            "asr": successes / len(eligible),
            "total_replacements": sum(
                r["n_replaced"] for r in records
            ),
        }

        split_results.append(summary)

        for r in detections:
            split_details.append({
                "split_seed": split_seed,
                "method": name,
                **r,
            })

        print(
            f"  {name:38s} "
            f"ASR={summary['asr']:.2%} "
            f"({successes}/{len(eligible)})"
        )

print("\nCompleted:", len(split_results), "method/split combinations")



Split 42 | Dictionary: 554 entries


42: guided: 100%|██████████| 200/200 [00:00<00:00, 1300.02it/s]


  guided                                 ASR=6.67% (13/195)


42: random_synonyms_guided_positions: 100%|██████████| 200/200 [00:00<00:00, 1331.59it/s]


  random_synonyms_guided_positions       ASR=5.64% (11/195)


42: fully_random: 100%|██████████| 200/200 [00:00<00:00, 1350.55it/s]


... (51 outputs left out)

108: random_synonyms_guided_positions: 100%|██████████| 200/200 [00:00<00:00, 1319.17it/s]


  random_synonyms_guided_positions       ASR=5.67% (11/194)


108: fully_random: 100%|██████████| 200/200 [00:00<00:00, 1324.54it/s]

  fully_random                           ASR=4.64% (9/194)

Completed: 30 method/split combinations


In [33]:

summary_path = OUT_DIR / "multisplit_summary_r0.1.jsonl"
details_path = OUT_DIR / "multisplit_details_r0.1.jsonl"

assert not summary_path.exists()
assert not details_path.exists()

save_jsonl(split_results, summary_path)
save_jsonl(split_details, details_path)

print("Saved:", summary_path.name)
print("Saved:", details_path.name)
print("Summary rows:", len(split_results))
print("Detail rows:", len(split_details))


Saved 30 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided/multisplit_summary_r0.1.jsonl
Saved 6000 records to /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/stealing_guided/multisplit_details_r0.1.jsonl
Saved: multisplit_summary_r0.1.jsonl
Saved: multisplit_details_r0.1.jsonl
Summary rows: 30
Detail rows: 6000


In [34]:

print("=== MULTI-SPLIT ROBUSTNESS (10 SPLITS) ===")

for method in [
    "guided",
    "random_synonyms_guided_positions",
    "fully_random",
]:
    values = np.array([
        r["asr"]
        for r in split_results
        if r["method"] == method
    ])

    print(f"\n{method}")
    print(f"Mean ASR: {values.mean():.2%}")
    print(f"Std ASR:  {values.std(ddof=1):.2%}")
    print(f"Min ASR:  {values.min():.2%}")
    print(f"Max ASR:  {values.max():.2%}")

guided_asr = {
    r["split_seed"]: r["asr"]
    for r in split_results if r["method"] == "guided"
}
fully_random_asr = {
    r["split_seed"]: r["asr"]
    for r in split_results if r["method"] == "fully_random"
}

differences = np.array([
    guided_asr[s] - fully_random_asr[s]
    for s in SPLIT_SEEDS
])

print("\nGuided - Fully Random:")
print(f"Mean difference: {differences.mean():+.2%}")
print(f"Positive splits: {(differences > 0).sum()}/10")


=== MULTI-SPLIT ROBUSTNESS (10 SPLITS) ===

guided
Mean ASR: 6.45%
Std ASR:  1.79%
Min ASR:  4.06%
Max ASR:  8.90%

random_synonyms_guided_positions
Mean ASR: 6.66%
Std ASR:  1.93%
Min ASR:  4.10%
Max ASR:  9.95%

fully_random
Mean ASR: 4.59%
Std ASR:  1.87%
Min ASR:  0.51%
Max ASR:  7.22%

Guided - Fully Random:
Mean difference: +1.86%
Positive splits: 9/10
